In [1]:
%%writefile lab9_npc.py
"""lab9_npc.py - offline stand-in model and game data for Lab 9 (LangChain NPC).

The class StandInNPC is a real LangChain chat model, so prompts, chains, parsers,
tools and message history all work on it exactly as they would on a hosted model.
It is NOT a language model: it uses keyword rules and seeded random numbers so that
every student gets the same output. Do not edit this file.

To use a real model later, replace the single line  llm = StandInNPC()  with, for
example,  llm = ChatOllama(model="llama3.2:3b")  or  llm = ChatGroq(model=...).
"""
import random
import re

from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool

LOG = []                      # every model call: prompt text and reply
_STATE = {"rng": random.Random(0), "mode": "normal"}


def set_seed(seed, mode="normal"):
    """Seed the stand-in. mode='stuck' makes it request a tool forever."""
    _STATE["rng"] = random.Random(seed)
    _STATE["mode"] = mode


# ------------------------------------------------------------------ game data
SHOP = {"iron sword": (2, 50), "health potion": (0, 15), "oak shield": (1, 40)}


@tool
def check_inventory(item: str) -> str:
    """Check how many of an item the shop has in stock and its price in gold."""
    stock, price = SHOP.get(item.lower(), (0, 0))
    if item.lower() not in SHOP:
        return f"{item}: not sold here"
    return f"{item.lower()}: {stock} in stock, {price} gold"


@tool
def roll_dice(sides: int = 20) -> str:
    """Roll one die with the given number of sides and return the number."""
    return str(_STATE["rng"].randint(1, int(sides)))


# ------------------------------------------------------- stand-in behaviour
PREFIX = {"grumpy": "*grunts* ", "cheerful": "Ha! ", "nervous": "Um... "}
EMOTION = {"grumpy": "angry", "cheerful": "happy", "nervous": "afraid"}


def _persona(system):
    m = re.search(r"You are (\w+), an? ([^.]+)\. Mood: (\w+)", system)
    return (m.group(1), m.group(2), m.group(3)) if m else ("Stranger", "villager", "calm")


def _find(texts, pattern):
    found = None
    for t in texts:
        m = re.search(pattern, t, re.I)
        if m:
            found = m.group(1)
    return found


def _line(system, humans, text):
    name, role, mood = _persona(system)
    pre, low = PREFIX.get(mood, ""), text.lower()
    if "what is my name" in low:
        who = _find(humans, r"my name is (\w+)")
        return pre + (f"You told me your name is {who}." if who else "You have not told me your name yet.")
    if "what did i" in low:
        what = _find(humans, r"i (?:chose|decided) to ([^.!?]+)")
        return pre + (f"You chose to {what}. I remember." if what else "You have not told me of any choice.")
    if "my name is" in low:
        return pre + f"Well met, {_find([text], r'my name is (\w+)')}. I am {name} the {role}."
    if re.search(r"i (chose|decided) to", low):
        return pre + "A bold choice. I will remember it."
    if "quest" in low or "work" in low:
        return pre + "Wolves are raiding the mill. Bring me three pelts."
    return pre + f"I am {name} the {role}. What do you need?"


def _item_in(text):
    for item in SHOP:
        if item in text or item.split()[-1] in text:
            return item
    return "iron sword"


def _respond(messages, tools):
    rng = _STATE["rng"]
    system = next((m.content for m in messages if m.type == "system"), "")
    humans = [m.content for m in messages if m.type == "human"]
    last = messages[-1]
    pre = PREFIX.get(_persona(system)[2], "")
    n_calls = sum(1 for m in messages if m.type == "ai" and getattr(m, "tool_calls", None))

    if tools and _STATE["mode"] == "stuck":
        return {"name": "roll_dice", "args": {"sides": 20}, "id": f"call_{n_calls}"}
    if last.type == "tool":
        if last.name == "check_inventory":
            if "0 in stock" in last.content:
                return pre + "Sold out. Come back after the next delivery."
            return pre + f"Yes. {last.content}."
        if last.name == "roll_dice":
            n = int(last.content)
            return pre + f"You rolled {n}. " + ("A solid hit!" if n >= 11 else "You miss.")
        return pre + "I cannot do that."
    text = humans[-1] if humans else ""
    low = text.lower()
    if tools:
        if re.search(r"\b(buy|price|cost|stock|sword|shield|potion)\b", low):
            if rng.random() < 0.15:
                return {"name": "forge_item", "args": {"item": _item_in(low)}, "id": f"call_{n_calls}"}
            return {"name": "check_inventory", "args": {"item": _item_in(low)}, "id": f"call_{n_calls}"}
        if re.search(r"\b(roll|attack|dice)\b", low):
            return {"name": "roll_dice", "args": {"sides": 20}, "id": f"call_{n_calls}"}
    if not system:
        return "Hello from the Lab 9 stand-in. I am a Python function, not a language model."
    line = _line(system, humans, text)
    if "JSON" in system:
        mood = _persona(system)[2]
        reply = ('{"say": "%s", "emotion": "%s", "action": "%s"}'
                 % (line.replace('"', "'"), EMOTION.get(mood, "neutral"),
                    "offer_quest" if "quest" in low else "none"))
        r = rng.random()
        if r < 0.25:
            return "Here is the reply: " + reply + " Hope this helps!"
        if r < 0.40:
            return reply.replace(', "action": "none"', "").replace(', "action": "offer_quest"', "")
        return reply
    return line


class StandInNPC(BaseChatModel):
    """Offline stand-in chat model. Keyword rules plus seeded randomness."""

    @property
    def _llm_type(self):
        return "stand-in-npc"

    def bind_tools(self, tools, **kwargs):
        return self.bind(tools=[convert_to_openai_tool(t) for t in tools], **kwargs)

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        reply = _respond(messages, kwargs.get("tools"))
        LOG.append({"prompt": "\n".join(f"{m.type}: {m.content}" for m in messages), "reply": reply})
        if isinstance(reply, str):
            msg = AIMessage(content=reply)
        else:
            msg = AIMessage(content="", tool_calls=[{**reply, "type": "tool_call"}])
        return ChatResult(generations=[ChatGeneration(message=msg)])


llm = StandInNPC()


Overwriting lab9_npc.py


In [2]:
# Setup
import json
import inspect

import langchain
import langchain_core
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.exceptions import OutputParserException
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda
from pydantic import BaseModel
from typing import Literal

import lab9_npc as npc
from lab9_npc import llm, set_seed, check_inventory, roll_dice

print("langchain-core", langchain_core.__version__, "| langchain", langchain.__version__)


langchain-core 1.8.4 | langchain 1.5.2


In [7]:
# Task 1 - Personas as template variables
PERSONAS = [
    {"npc": "Brom", "role": "gruff blacksmith", "mood": "grumpy"},
    {"npc": "Lyra", "role": "village healer", "mood": "cheerful"},
    {"npc": "Pip", "role": "shy messenger", "mood": "nervous"},
    {"npc": "Mira", "role": "old fisherman", "mood": "cheerful"},   # TODO 2: my own fourth persona
]
for p in PERSONAS:
    print(f"{p['npc']:5} ({p['mood']:8}):", npc_chain.invoke({**p, "player": "Do you have any work for me?"}))


Brom  (grumpy  ): *grunts* Wolves are raiding the mill. Bring me three pelts.
Lyra  (cheerful): Ha! Wolves are raiding the mill. Bring me three pelts.
Pip   (nervous ): Um... Wolves are raiding the mill. Bring me three pelts.
Mira  (cheerful): Ha! Wolves are raiding the mill. Bring me three pelts.


In [8]:
# Task 2 - Structured replies, parsed and retried
class NPCReply(BaseModel):
    say: str
    emotion: Literal["angry", "happy", "afraid", "neutral"]
    action: Literal["none", "offer_quest", "open_shop"]

parser = PydanticOutputParser(pydantic_object=NPCReply)
json_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM + "\n{format_instructions}"), ("human", "{player}")
]).partial(format_instructions=parser.get_format_instructions())
json_chain = json_prompt | llm | parser
safe_chain = json_chain.with_retry(retry_if_exception_type=(OutputParserException,),
                                   stop_after_attempt=3, wait_exponential_jitter=False)

def measure(chain, seeds=range(20)):
    ok = failed = 0
    npc.LOG.clear()
    for seed in seeds:
        set_seed(seed)
        for x in lines:
            try:
                chain.invoke({**BROM, "player": x})
                ok += 1
            except OutputParserException:
                failed += 1
    return ok, failed, len(npc.LOG)

for seed in range(4):
    set_seed(seed)
    try:
        print(f"seed {seed}: accepted ->", json_chain.invoke({**BROM, "player": "Do you have any quest?"}))
    except OutputParserException as err:
        print(f"seed {seed}: REJECTED ->", str(err).splitlines()[0][:60])
for name, chain in [("no retry", json_chain), ("with retry", safe_chain)]:
    ok, failed, calls = measure(chain)
    print(f"{name:10}: usable {ok}/60 ({ok / 60:.0%}), failed {failed}, model calls {calls}")


seed 0: accepted -> say='*grunts* Wolves are raiding the mill. Bring me three pelts.' emotion='angry' action='offer_quest'
seed 1: REJECTED -> Invalid json output: Here is the reply: {"say": "*grunts* Wo
seed 2: accepted -> say='*grunts* Wolves are raiding the mill. Bring me three pelts.' emotion='angry' action='offer_quest'
seed 3: REJECTED -> Invalid json output: Here is the reply: {"say": "*grunts* Wo
no retry  : usable 41/60 (68%), failed 19, model calls 60
with retry: usable 59/60 (98%), failed 1, model calls 90


In [9]:
# Task 3 - Tools and a safe agent loop
TOOLS = {"check_inventory": check_inventory, "roll_dice": roll_dice}

def tool_npc(player_text, max_steps=4):
    messages = [SystemMessage(content=SYSTEM.format(**BROM)), HumanMessage(content=player_text)]
    bound = llm.bind_tools(list(TOOLS.values()))
    trace = []
    for step in range(1, max_steps + 1):
        ai = bound.invoke(messages)
        messages.append(ai)
        if not ai.tool_calls:
            return ai.content, trace + [f"step {step}: final answer"]
        for call in ai.tool_calls:
            fn = TOOLS.get(call["name"])
            if fn is None:
                result = f"unknown tool {call['name']}. Allowed: {', '.join(TOOLS)}"
            else:
                result = fn.invoke(call["args"])
            trace.append(f"step {step}: {call['name']}({call['args']}) -> {result}")
            messages.append(ToolMessage(content=str(result), tool_call_id=call["id"], name=call["name"]))
    return None, trace + ["stopped: step limit reached"]

set_seed(2)
for q in ["How much is the iron sword?", "I attack the goblin!", "Can I buy a health potion?"]:
    answer, trace = tool_npc(q)
    print("Player:", q)
    print("\n".join("  " + t for t in trace))
    print("  Brom:", answer)

counts = {"answered": 0, "refused unknown tool": 0, "step limit": 0}
for seed in range(40):
    set_seed(seed)
    answer, trace = tool_npc("How much is the iron sword?")
    counts["answered" if answer else "step limit"] += 1
    counts["refused unknown tool"] += any("unknown tool" in t for t in trace)
print("40 runs:", counts)
set_seed(0, mode="stuck")
answer, trace = tool_npc("Hello")
print("stuck model:", answer, "|", trace[-1], "| tool calls made:", len(trace) - 1)


Player: How much is the iron sword?
  step 1: check_inventory({'item': 'iron sword'}) -> iron sword: 2 in stock, 50 gold
  step 2: final answer
  Brom: *grunts* Yes. iron sword: 2 in stock, 50 gold.
Player: I attack the goblin!
  step 1: roll_dice({'sides': 20}) -> 2
  step 2: final answer
  Brom: *grunts* You rolled 2. You miss.
Player: Can I buy a health potion?
  step 1: forge_item({'item': 'health potion'}) -> unknown tool forge_item. Allowed: check_inventory, roll_dice
  step 2: final answer
  Brom: *grunts* I cannot do that.
40 runs: {'answered': 40, 'refused unknown tool': 5, 'step limit': 0}
stuck model: None | stopped: step limit reached | tool calls made: 4


In [10]:
# Task 4 - Memory across turns
from langchain_core.messages import AIMessage

mem_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM), MessagesPlaceholder("history"), ("human", "{player}")])
mem_chain = mem_prompt | llm | StrOutputParser()
HISTORY = {}                       # one list of messages per game session

def say(session, text):
    history = HISTORY.setdefault(session, [])
    reply = mem_chain.invoke({**BROM, "player": text, "history": history})
    history += [HumanMessage(content=text), AIMessage(content=reply)]
    return reply

for s, text in [("game1", "My name is Aria."), ("game1", "I decided to spare the bandit."),
                ("game1", "What is my name?"), ("game1", "What did I decide?"),
                ("game2", "What is my name?")]:
    print(f"[{s}] Player: {text}\n        Brom: {say(s, text)}")
print("messages stored in game1:", len(HISTORY["game1"]), "| in game2:", len(HISTORY["game2"]))


[game1] Player: My name is Aria.
        Brom: *grunts* Well met, Aria. I am Brom the gruff blacksmith.
[game1] Player: I decided to spare the bandit.
        Brom: *grunts* A bold choice. I will remember it.
[game1] Player: What is my name?
        Brom: *grunts* You told me your name is Aria.
[game1] Player: What did I decide?
        Brom: *grunts* You chose to spare the bandit. I remember.
[game2] Player: What is my name?
        Brom: *grunts* You have not told me your name yet.
messages stored in game1: 8 | in game2: 2


In [11]:
# Task 5 - The companion: routing, tools and memory together
def needs_tool(text):
    low = text.lower()
    return any(w in low for w in ["buy", "price", "cost", "sword", "potion", "shield", "attack", "roll"])

def route(inputs):
    if needs_tool(inputs["player"]):
        return RunnableLambda(lambda i: tool_npc(i["player"])[0])
    return RunnableLambda(lambda i: say(i["session"], i["player"]))

companion = RunnableLambda(route)
SCRIPT = [("My name is Aria.", "Aria"), ("I chose to guard the village.", "remember"),
          ("How much is the iron sword?", "gold"), ("I attack the wolf!", "rolled"),
          ("What is my name?", "Aria"), ("What did I decide?", "guard the village")]

HISTORY.clear(); npc.LOG.clear(); set_seed(5)
passed = 0
for text, must_contain in SCRIPT:
    out = companion.invoke({"player": text, "session": "run1"}) or ""
    ok = must_contain in out
    passed += ok
    print(f"{'PASS' if ok else 'FAIL'} | {text:30} -> {out}")
print(f"checks passed: {passed}/{len(SCRIPT)} | model calls: {len(npc.LOG)}")


PASS | My name is Aria.               -> *grunts* Well met, Aria. I am Brom the gruff blacksmith.
PASS | I chose to guard the village.  -> *grunts* A bold choice. I will remember it.
PASS | How much is the iron sword?    -> *grunts* Yes. iron sword: 2 in stock, 50 gold.
PASS | I attack the wolf!             -> *grunts* You rolled 12. A solid hit!
PASS | What is my name?               -> *grunts* You told me your name is Aria.
PASS | What did I decide?             -> *grunts* You chose to guard the village. I remember.
checks passed: 6/6 | model calls: 8


In [12]:
# Task 5b - Measure: hand-rolled versus framework
def hand_json(player_text, tries=3):
    system = SYSTEM.format(**BROM) + "\nReply with JSON: " +'{"say": "...", "emotion": "...", "action": "..."}'
    for _ in range(tries):
        raw = llm.invoke([SystemMessage(content=system), HumanMessage(content=player_text)]).content
        try:
            return NPCReply(**json.loads(raw))
        except (json.JSONDecodeError, ValueError):   # ValueError also covers pydantic ValidationError
            continue
    return None

npc.LOG.clear()
ok = 0
for seed in range(20):
    set_seed(seed)
    for x in lines:
        ok += hand_json(x) is not None
print(f"hand-rolled : usable {ok}/60, model calls {len(npc.LOG)}, source lines {len(inspect.getsource(hand_json).splitlines())}")
ok, failed, calls = measure(safe_chain)
print(f"framework   : usable {ok}/60, model calls {calls}")
print("framework also gives: batch ->", hasattr(safe_chain, "batch"),
      "| stream ->", hasattr(safe_chain, "stream"), "| swap the model by changing one line")


hand-rolled : usable 59/60, model calls 90, source lines 9


framework   : usable 59/60, model calls 90


framework also gives: batch -> True | stream -> True | swap the model by changing one line


## Task 6 - Figures for the worksheet (measured, not copied)

In [13]:
# Task 6 - collect my own figures for the worksheet
def n_lines(obj): return len(inspect.getsource(obj).splitlines())

# model calls for one request of each simple method
npc.LOG.clear(); hand_npc("Hello there.");                 c_hand = len(npc.LOG)
npc.LOG.clear(); npc_chain.invoke({**BROM, "player": "Hello there."}); c_chain = len(npc.LOG)

print("hand_npc source lines      :", n_lines(hand_npc), "| calls per request:", c_hand)
print("chain (prompt|llm|parser)  : 2 lines (prompt + npc_chain)       | calls per request:", c_chain)
ok, failed, calls = measure(safe_chain)
print("safe_chain (Task 2)        : 5 lines (class excluded)          | usable", ok, "/60, calls", calls, "-> avg", round(calls/60, 2), "calls/request")
ok2 = 0; npc.LOG.clear()
for seed in range(20):
    set_seed(seed)
    for x in lines: ok2 += hand_json(x) is not None
print("hand_json (Task 5)         :", n_lines(hand_json), "lines | usable", ok2, "/60, calls", len(npc.LOG))
print("companion (Task 5)         : route+needs_tool", n_lines(route)+n_lines(needs_tool), "lines (+ tool_npc", n_lines(tool_npc), "+ say", n_lines(say), ") | 8 calls for 6 lines")


hand_npc source lines      : 4 | calls per request: 1
chain (prompt|llm|parser)  : 2 lines (prompt + npc_chain)       | calls per request: 1


safe_chain (Task 2)        : 5 lines (class excluded)          | usable 59 /60, calls 90 -> avg 1.5 calls/request
hand_json (Task 5)         : 9 lines | usable 59 /60, calls 90
companion (Task 5)         : route+needs_tool 7 lines (+ tool_npc 18 + say 5 ) | 8 calls for 6 lines
